# Exam (Medium) — Practice Problems

**Format:** Two tiers of practice, matching real CoderPad dynamics.

**Part A: Warmups (8 problems, 5-10 min each)**
Fundamentals that must be automatic.

**Part B: Full Problems (8 problems, 30 min each)**
Set a timer, no docs, no autocomplete. Every problem gives you the full spec.

**Mix:** 50% data pipelines / 37.5% classic ML basics / 12.5% basic training

**Rules:**
- Set a timer per problem
- No docs, no autocomplete, no outside help
- Run the test cell — all assertions must pass

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from typing import Any
from collections import Counter, defaultdict
import math

---

# Part A: Warmups (5-10 min each)

These must be automatic. If any take > 10 min, that's priority #1 to drill.

---

## Warmup 1: Stable Softmax

Implement softmax that doesn't overflow on large inputs.

In [ ]:
def stable_softmax(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    """Numerically stable softmax. No torch.softmax or F.softmax."""
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Warmup 1 ---
x = torch.tensor([1.0, 2.0, 3.0])
s = stable_softmax(x)
assert torch.allclose(s, F.softmax(x, dim=-1), atol=1e-5)
assert abs(s.sum().item() - 1.0) < 1e-5, "Should sum to 1"

# Stability test — this would overflow with naive exp
big = torch.tensor([1000.0, 1001.0, 1002.0])
s_big = stable_softmax(big)
assert not torch.isnan(s_big).any(), "Should not produce NaN on large inputs"
assert not torch.isinf(s_big).any(), "Should not produce Inf on large inputs"
assert abs(s_big.sum().item() - 1.0) < 1e-5

# Batch
batch = torch.randn(4, 10)
s_batch = stable_softmax(batch, dim=-1)
assert s_batch.shape == (4, 10)
assert torch.allclose(s_batch.sum(dim=-1), torch.ones(4), atol=1e-5)

print("Warmup 1: PASSED")

---

## Warmup 2: Cosine Similarity Matrix

Given two batches of vectors, compute the full pairwise cosine similarity matrix.

In [ ]:
def cosine_similarity_matrix(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """Pairwise cosine similarity. a: (N, D), b: (M, D) -> (N, M). Handle zero vectors."""
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Warmup 2 ---
a = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
b = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
sim = cosine_similarity_matrix(a, b)
assert sim.shape == (3, 2)
assert abs(sim[0, 0].item() - 1.0) < 1e-5  # identical
assert abs(sim[0, 1].item() - 0.0) < 1e-5  # orthogonal
assert abs(sim[2, 0].item() - 1/math.sqrt(2)) < 1e-5

# Zero vector
c = torch.tensor([[0.0, 0.0], [1.0, 0.0]])
sim_z = cosine_similarity_matrix(c, b)
assert sim_z[0, 0].item() == 0.0, "Zero vector sim should be 0"
assert sim_z[0, 1].item() == 0.0

# Larger batch
a_big = torch.randn(100, 64)
b_big = torch.randn(50, 64)
sim_big = cosine_similarity_matrix(a_big, b_big)
assert sim_big.shape == (100, 50)
assert sim_big.min() >= -1.0 - 1e-5
assert sim_big.max() <= 1.0 + 1e-5

print("Warmup 2: PASSED")

---

## Warmup 3: One-Hot Encoding

Convert integer class labels to one-hot vectors. No `F.one_hot`.

In [ ]:
def one_hot(labels: torch.Tensor, num_classes: int) -> torch.Tensor:
    """labels: (B,) int64 -> (B, num_classes) float32 one-hot. No F.one_hot."""
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Warmup 3 ---
labels = torch.tensor([0, 2, 1, 4])
oh = one_hot(labels, num_classes=5)
assert oh.shape == (4, 5)
assert oh.dtype == torch.float32
assert oh[0].tolist() == [1, 0, 0, 0, 0]
assert oh[1].tolist() == [0, 0, 1, 0, 0]
assert oh[3].tolist() == [0, 0, 0, 0, 1]
assert (oh.sum(dim=-1) == 1).all(), "Each row should sum to 1"

# Single element
oh_single = one_hot(torch.tensor([0]), 3)
assert oh_single.shape == (1, 3)

print("Warmup 3: PASSED")

---

## Warmup 4: Cross-Entropy Loss

Implement cross-entropy loss from scratch given logits and integer labels. No `F.cross_entropy`.

In [ ]:
def cross_entropy(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """logits: (B, C), targets: (B,) int64. Return scalar mean loss. No F.cross_entropy.
    Must be numerically stable (use log-sum-exp trick)."""
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Warmup 4 ---
logits = torch.tensor([[2.0, 1.0, 0.1], [0.5, 2.5, 0.3]])
targets = torch.tensor([0, 1])
loss = cross_entropy(logits, targets)
expected = F.cross_entropy(logits, targets)
assert abs(loss.item() - expected.item()) < 1e-5, f"Expected {expected.item()}, got {loss.item()}"

# Numerical stability
big_logits = torch.tensor([[1000.0, 999.0], [0.0, 1000.0]])
big_targets = torch.tensor([0, 1])
big_loss = cross_entropy(big_logits, big_targets)
assert not torch.isnan(big_loss), "Should handle large logits"
assert not torch.isinf(big_loss), "Should handle large logits"

# Gradient flows
logits_g = torch.randn(4, 5, requires_grad=True)
targets_g = torch.randint(0, 5, (4,))
loss_g = cross_entropy(logits_g, targets_g)
loss_g.backward()
assert logits_g.grad is not None

print("Warmup 4: PASSED")

---

## Warmup 5: Confusion Matrix

Build a confusion matrix from predictions and labels. No sklearn.

In [ ]:
def confusion_matrix(predictions: torch.Tensor, targets: torch.Tensor, num_classes: int) -> torch.Tensor:
    """Return (num_classes, num_classes) matrix where cm[true][pred] = count. No sklearn."""
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Warmup 5 ---
preds = torch.tensor([0, 1, 2, 0, 1, 2, 0, 0])
targs = torch.tensor([0, 1, 2, 1, 0, 2, 0, 2])
cm = confusion_matrix(preds, targs, num_classes=3)
assert cm.shape == (3, 3)
assert cm.sum().item() == 8, "Total should equal number of samples"

# Diagonal = correct predictions
assert cm[0, 0].item() == 2  # true=0, pred=0
assert cm[1, 1].item() == 1  # true=1, pred=1
assert cm[2, 2].item() == 2  # true=2, pred=2

# Off-diagonal = errors
assert cm[1, 0].item() == 1  # true=1, pred=0
assert cm[0, 1].item() == 1  # true=0, pred=1
assert cm[2, 0].item() == 1  # true=2, pred=0

# Row sums = number of true instances per class
assert cm[0].sum().item() == 3  # 3 true class-0 samples
assert cm[1].sum().item() == 2  # 2 true class-1 samples
assert cm[2].sum().item() == 3  # 3 true class-2 samples

print("Warmup 5: PASSED")

---

## Warmup 6: Pairwise L2 Distances

Given two tensors `a` of shape `(N, D)` and `b` of shape `(M, D)`, compute a pairwise L2 distance matrix of shape `(N, M)` where entry `(i, j)` is the Euclidean distance between `a[i]` and `b[j]`.

**Formula (expanded form, avoids loops):**

$$\|a_i - b_j\|_2^2 = \|a_i\|^2 + \|b_j\|^2 - 2 \cdot a_i \cdot b_j$$

Then take the square root. Clamp before sqrt to avoid negative values from floating-point errors.

**Constraints:**
- No loops. Use broadcasting and matrix multiplication.
- Return float32 tensor of shape `(N, M)`.

In [ ]:
def pairwise_l2(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """Compute pairwise L2 distance matrix.

    Args:
        a: (N, D) tensor
        b: (M, D) tensor

    Returns:
        (N, M) distance matrix.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Warmup 6 ---
torch.manual_seed(42)

# Basic case
a = torch.tensor([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0]])
b = torch.tensor([[1.0, 1.0], [0.0, 0.0]])
D = pairwise_l2(a, b)
assert D.shape == (3, 2)
assert torch.allclose(D[0, 0], torch.tensor(math.sqrt(2)), atol=1e-5)  # (0,0)->(1,1)
assert torch.allclose(D[0, 1], torch.tensor(0.0), atol=1e-5)          # (0,0)->(0,0)
assert torch.allclose(D[1, 0], torch.tensor(1.0), atol=1e-5)          # (1,0)->(1,1)
assert torch.allclose(D[2, 0], torch.tensor(1.0), atol=1e-5)          # (0,1)->(1,1)

# Distance to self is zero
x = torch.randn(5, 10)
D_self = pairwise_l2(x, x)
assert D_self.shape == (5, 5)
assert torch.allclose(D_self.diag(), torch.zeros(5), atol=1e-5)

# Symmetry: D(a,b) = D(b,a).T
a2 = torch.randn(4, 8)
b2 = torch.randn(6, 8)
assert torch.allclose(pairwise_l2(a2, b2), pairwise_l2(b2, a2).T, atol=1e-4)

# All distances non-negative
assert (pairwise_l2(a2, b2) >= 0).all()

# dtype check
assert D.dtype == torch.float32

print("Warmup 6: PASSED")

---

## Warmup 7: Decision Tree Split (Information Gain)

Given a dataset with binary features and labels, find the best feature to split on using information gain.

**Formulas:**

Entropy: $H(S) = -\sum_c p_c \log_2(p_c)$ (0 if $p_c = 0$)

Information Gain: $IG(S, feature) = H(S) - \sum_{v \in \{0,1\}} \frac{|S_v|}{|S|} H(S_v)$

where $S_v$ is the subset of S where the feature has value v.

In [ ]:
def entropy(labels: torch.Tensor) -> float:
    """Compute entropy of a label tensor. Labels are integer class indices."""
    # YOUR CODE HERE
    pass


def best_split(features: torch.Tensor, labels: torch.Tensor) -> dict:
    """Find the best binary feature to split on.

    Args:
        features: (N, F) binary feature matrix (0/1 values).
        labels: (N,) integer class labels.

    Returns:
        Dict with 'feature_idx' (int), 'info_gain' (float).
    """
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Warmup 7 ---

# Perfect split: feature 0 perfectly separates classes
features = torch.tensor([[0, 1], [0, 0], [1, 1], [1, 0]], dtype=torch.float32)
labels = torch.tensor([0, 0, 1, 1])

# Entropy of full set: 2 class 0, 2 class 1 → H = 1.0
h = entropy(labels)
assert abs(h - 1.0) < 1e-5, f"Expected entropy=1.0, got {h}"

# Pure set: entropy = 0
assert abs(entropy(torch.tensor([0, 0, 0])) - 0.0) < 1e-5
assert abs(entropy(torch.tensor([1])) - 0.0) < 1e-5

# Best split
result = best_split(features, labels)
assert result["feature_idx"] == 0, f"Feature 0 is the perfect split, got {result['feature_idx']}"
assert abs(result["info_gain"] - 1.0) < 1e-5, f"Perfect split should have IG=1.0, got {result['info_gain']}"

# Feature 1 has zero gain (doesn't separate classes)
# With feature 1: split into {(0,1),(1,1)} and {(0,0),(1,0)} → both have 1 of each class → IG = 0
features2 = torch.tensor([[0, 0], [0, 0], [1, 1], [1, 1]], dtype=torch.float32)
labels2 = torch.tensor([0, 0, 1, 1])
result2 = best_split(features2, labels2)
assert result2["feature_idx"] == 0

# 3-class case
labels3 = torch.tensor([0, 0, 1, 1, 2, 2])
h3 = entropy(labels3)
expected_h3 = -3 * (2/6) * math.log2(2/6)  # = log2(3) ≈ 1.585
assert abs(h3 - expected_h3) < 1e-5

print("Warmup 7: PASSED")

---

## Warmup 8: SVM Hinge Loss

Compute the multi-class hinge loss (Weston-Watkins formulation) for a batch of scores and labels.

**Formula:**

For each sample $i$ with true class $y_i$:
$$L_i = \sum_{j \neq y_i} \max(0, s_j - s_{y_i} + \Delta)$$

Total loss: $L = \frac{1}{N} \sum_i L_i$

Where $\Delta$ (margin) is typically 1.0.

In [ ]:
def hinge_loss(scores: torch.Tensor, labels: torch.Tensor, margin: float = 1.0) -> torch.Tensor:
    """Multi-class hinge loss.

    Args:
        scores: (N, C) raw scores for each class.
        labels: (N,) integer class labels.
        margin: Margin parameter (default 1.0).

    Returns:
        Scalar loss tensor.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests: Warmup 8 ---

# Perfect scores: correct class has highest score by > margin → loss = 0
scores = torch.tensor([[10.0, 0.0, 0.0], [0.0, 10.0, 0.0]])
labels = torch.tensor([0, 1])
loss = hinge_loss(scores, labels, margin=1.0)
assert abs(loss.item()) < 1e-5, f"Perfect scores should give zero loss, got {loss.item()}"

# All same scores: loss = margin * (num_classes - 1)
scores_eq = torch.tensor([[1.0, 1.0, 1.0]])
labels_eq = torch.tensor([0])
loss_eq = hinge_loss(scores_eq, labels_eq, margin=1.0)
# For each wrong class j: max(0, 1 - 1 + 1) = 1, two wrong classes → 2.0
assert abs(loss_eq.item() - 2.0) < 1e-5, f"Expected 2.0, got {loss_eq.item()}"

# Wrong class has higher score
scores_wrong = torch.tensor([[0.0, 3.0]])
labels_wrong = torch.tensor([0])
loss_wrong = hinge_loss(scores_wrong, labels_wrong, margin=1.0)
# max(0, 3 - 0 + 1) = 4
assert abs(loss_wrong.item() - 4.0) < 1e-5, f"Expected 4.0, got {loss_wrong.item()}"

# Gradient flows
scores_g = torch.randn(4, 5, requires_grad=True)
labels_g = torch.randint(0, 5, (4,))
loss_g = hinge_loss(scores_g, labels_g)
loss_g.backward()
assert scores_g.grad is not None

# Batch: mean over samples
scores_batch = torch.tensor([[10.0, 0.0], [0.0, 0.0]])
labels_batch = torch.tensor([0, 1])
loss_batch = hinge_loss(scores_batch, labels_batch, margin=1.0)
# Sample 0: max(0, 0-10+1) = 0
# Sample 1: max(0, 0-0+1) = 1
# Mean: 0.5
assert abs(loss_batch.item() - 0.5) < 1e-5, f"Expected 0.5, got {loss_batch.item()}"

print("Warmup 8: PASSED")

---

# Part B: Full Problems (30 min each)

Set a 30-minute timer. No docs, no autocomplete.

---

## Problem 1: PyTorch Dataset with Transforms and Collation

### Scenario

Build a PyTorch Dataset for image-caption pairs with a custom collate function.

### Requirements

**`ImageCaptionDataset(Dataset)`:**
- `__init__(self, records: list[dict], transform=None)` — records have `"image"` (HxWx3 numpy uint8), `"caption"` (str)
- `__getitem__` returns `{"image": tensor, "tokens": tensor}` where:
  - image: float32, (3,H,W), values in [0,1] (divide by 255). Apply transform if provided.
  - tokens: int64 1D tensor from `tokenize(caption)` (provided below)
- `__len__` returns count

**`padded_collate(batch: list[dict]) -> dict`:**
- `"image"`: stack into (B,3,H,W) — assume same spatial size
- `"tokens"`: pad to longest sequence with 0, return (B, max_len)
- `"mask"`: attention mask (B, max_len) — 1 for real tokens, 0 for padding
- `"lengths"`: original sequence lengths as (B,) int64 tensor

```python
def tokenize(text: str) -> list[int]:
    return [hash(w) % 10000 + 1 for w in text.lower().split()]
```

In [ ]:
def tokenize(text: str) -> list[int]:
    return [hash(w) % 10000 + 1 for w in text.lower().split()]


class ImageCaptionDataset(Dataset):
    """PyTorch Dataset for image-caption pairs."""

    def __init__(self, records: list[dict], transform=None):
        # YOUR CODE HERE
        pass

    def __len__(self):
        # YOUR CODE HERE
        pass

    def __getitem__(self, idx):
        # YOUR CODE HERE
        pass


def padded_collate(batch: list[dict]) -> dict:
    """Custom collate function with padding for variable-length token sequences."""
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 1 ---
records = [
    {"image": np.random.randint(0, 256, (32, 32, 3), dtype=np.uint8), "caption": "a red car"},
    {"image": np.random.randint(0, 256, (32, 32, 3), dtype=np.uint8), "caption": "a beautiful sunset over the ocean"},
    {"image": np.random.randint(0, 256, (32, 32, 3), dtype=np.uint8), "caption": "dog"},
]

ds = ImageCaptionDataset(records)
assert len(ds) == 3

item = ds[0]
assert item["image"].shape == (3, 32, 32), f"Expected (3,32,32), got {item['image'].shape}"
assert item["image"].dtype == torch.float32
assert item["image"].min() >= 0.0 and item["image"].max() <= 1.0
assert item["tokens"].dtype == torch.int64
assert item["tokens"].ndim == 1
assert len(item["tokens"]) == 3  # "a red car" → 3 tokens

# With transform
def dummy_transform(img):
    return img * 2  # just scale

ds_t = ImageCaptionDataset(records, transform=dummy_transform)
item_t = ds_t[0]
assert item_t["image"].max() <= 2.0  # scaled by transform

# Collation
batch = [ds[i] for i in range(3)]
collated = padded_collate(batch)
assert collated["image"].shape == (3, 3, 32, 32)
assert collated["tokens"].shape[0] == 3
max_len = collated["tokens"].shape[1]
assert max_len == 6  # "a beautiful sunset over the ocean" → 6 tokens
assert collated["mask"].shape == (3, max_len)
assert collated["lengths"].tolist() == [3, 6, 1]

# Padding correctness
assert collated["mask"][2].sum().item() == 1  # "dog" → 1 token
assert (collated["tokens"][2, 1:] == 0).all()  # padded with 0
assert collated["mask"][1].sum().item() == 6  # longest, no padding

# DataLoader integration
loader = DataLoader(ds, batch_size=2, collate_fn=padded_collate)
batch_dl = next(iter(loader))
assert batch_dl["image"].shape[0] == 2

print("Problem 1: ALL TESTS PASSED")

---

## Problem 2: Embedding Retrieval with Metadata Filters

### Scenario

You're building a simple search system over a dataset of items, each with an embedding vector and metadata tags.

### Requirements

**`SearchIndex`:**
- `__init__(self, dim: int)` — embedding dimension
- `add(self, items: list[dict])` — each dict has `"id"` (str), `"embedding"` (list[float] or tensor of length dim), `"tags"` (set[str]). Can be called multiple times.
- `search(self, query_embedding, k: int, require_tags: set[str] | None = None, exclude_tags: set[str] | None = None) -> list[dict]`:
  - Find top-k items by **cosine similarity** to query_embedding
  - If `require_tags` is provided, only include items that have ALL of those tags
  - If `exclude_tags` is provided, exclude items that have ANY of those tags
  - Return list of `{"id": str, "score": float}` dicts, sorted by descending score
  - **Cosine similarity formula:** `cos(a, b) = (a · b) / (||a|| * ||b||)`. If either vector is zero, similarity = 0.
- `__len__` — total items in index

**Important:**
- Use matrix operations for the similarity computation (not loops over individual items)
- Apply tag filters BEFORE computing similarities (don't compute similarity for filtered-out items)
- If k > number of matching items after filtering, return all matching items

In [ ]:
class SearchIndex:
    """Embedding-based search index with metadata filtering."""

    def __init__(self, dim: int):
        # YOUR CODE HERE
        pass

    def add(self, items: list[dict]) -> None:
        # YOUR CODE HERE
        pass

    def search(self, query_embedding, k: int, require_tags: set[str] | None = None, exclude_tags: set[str] | None = None) -> list[dict]:
        # YOUR CODE HERE
        pass

    def __len__(self) -> int:
        # YOUR CODE HERE
        pass

In [ ]:
# --- Tests for Problem 2 ---
idx = SearchIndex(dim=4)
assert len(idx) == 0

idx.add([
    {"id": "cat1", "embedding": [1.0, 0.0, 0.0, 0.0], "tags": {"animal", "indoor"}},
    {"id": "cat2", "embedding": [0.9, 0.1, 0.0, 0.0], "tags": {"animal", "outdoor"}},
    {"id": "car1", "embedding": [0.0, 1.0, 0.0, 0.0], "tags": {"vehicle", "outdoor"}},
    {"id": "car2", "embedding": [0.0, 0.9, 0.1, 0.0], "tags": {"vehicle", "indoor"}},
    {"id": "tree", "embedding": [0.0, 0.0, 1.0, 0.0], "tags": {"nature", "outdoor"}},
])
assert len(idx) == 5

# Basic search — query close to [1,0,0,0]
results = idx.search([1.0, 0.0, 0.0, 0.0], k=2)
assert len(results) == 2
assert results[0]["id"] == "cat1"
assert abs(results[0]["score"] - 1.0) < 1e-5
assert results[1]["id"] == "cat2"  # most similar after cat1

# Search with require_tags
results_outdoor = idx.search([1.0, 0.0, 0.0, 0.0], k=3, require_tags={"outdoor"})
result_ids = [r["id"] for r in results_outdoor]
assert "cat1" not in result_ids, "cat1 is indoor, should be filtered out"
assert "cat2" in result_ids

# Search with exclude_tags
results_no_vehicle = idx.search([0.0, 1.0, 0.0, 0.0], k=5, exclude_tags={"vehicle"})
result_ids_nv = [r["id"] for r in results_no_vehicle]
assert "car1" not in result_ids_nv
assert "car2" not in result_ids_nv

# Both filters together
results_both = idx.search([1.0, 0.0, 0.0, 0.0], k=5, require_tags={"outdoor"}, exclude_tags={"vehicle"})
result_ids_both = {r["id"] for r in results_both}
assert "car1" not in result_ids_both  # excluded by vehicle
assert "cat1" not in result_ids_both  # excluded by not outdoor
assert "cat2" in result_ids_both  # outdoor + animal

# k > matching items
results_big_k = idx.search([1.0, 0.0, 0.0, 0.0], k=100)
assert len(results_big_k) == 5

# Incremental add
idx.add([{"id": "dog", "embedding": [0.8, 0.2, 0.0, 0.0], "tags": {"animal", "outdoor"}}])
assert len(idx) == 6

# Results are sorted descending by score
results_sorted = idx.search([1.0, 0.0, 0.0, 0.0], k=6)
scores = [r["score"] for r in results_sorted]
assert scores == sorted(scores, reverse=True), "Results should be sorted by descending score"

print("Problem 2: ALL TESTS PASSED")

---

## Problem 3: Video Clip Processor

### Scenario

You have a list of video clip metadata dicts. Build functions to filter, validate, and batch them.

### Requirements

**`validate_clips(clips: list[dict]) -> tuple[list[dict], list[dict]]`:**
- Each clip dict has keys: `"id"` (str), `"duration"` (float, seconds), `"resolution"` (tuple[int,int] as (H,W)), `"fps"` (float), `"caption"` (str), `"quality_score"` (float 0-1)
- A clip is **valid** if ALL of:
  - duration is between 1.0 and 30.0 seconds (inclusive)
  - resolution H and W are both >= 256
  - fps >= 15.0
  - caption is non-empty (after stripping whitespace)
  - quality_score >= 0.5
- Return `(valid_clips, rejected_clips)`

**`batch_clips(clips: list[dict], max_batch_duration: float) -> list[list[dict]]`:**
- Greedily pack clips into batches where total duration per batch <= max_batch_duration
- Process clips in the order given — don't reorder
- A clip that exceeds max_batch_duration by itself goes in its own batch
- Return list of batches (each batch is a list of clip dicts)

**`compute_batch_stats(batches: list[list[dict]]) -> list[dict]`:**
- For each batch, return a dict with:
  - `"num_clips"`: int
  - `"total_duration"`: float
  - `"avg_quality"`: float (mean quality_score)
  - `"resolutions"`: set of unique (H,W) tuples in the batch

In [ ]:
def validate_clips(clips: list[dict]) -> tuple[list[dict], list[dict]]:
    """Validate video clips against quality criteria."""
    # YOUR CODE HERE
    pass


def batch_clips(clips: list[dict], max_batch_duration: float) -> list[list[dict]]:
    """Greedily pack clips into duration-limited batches."""
    # YOUR CODE HERE
    pass


def compute_batch_stats(batches: list[list[dict]]) -> list[dict]:
    """Compute summary statistics for each batch."""
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 3 ---
clips = [
    {"id": "a", "duration": 5.0, "resolution": (720, 1280), "fps": 30.0, "caption": "a red car driving", "quality_score": 0.9},
    {"id": "b", "duration": 0.5, "resolution": (720, 1280), "fps": 30.0, "caption": "too short", "quality_score": 0.8},  # duration < 1
    {"id": "c", "duration": 3.0, "resolution": (128, 128), "fps": 30.0, "caption": "too small", "quality_score": 0.7},  # resolution < 256
    {"id": "d", "duration": 10.0, "resolution": (512, 512), "fps": 24.0, "caption": "a sunset", "quality_score": 0.85},
    {"id": "e", "duration": 8.0, "resolution": (1080, 1920), "fps": 60.0, "caption": "  ", "quality_score": 0.95},  # empty caption
    {"id": "f", "duration": 4.0, "resolution": (480, 640), "fps": 30.0, "caption": "dog playing", "quality_score": 0.3},  # low quality
    {"id": "g", "duration": 7.0, "resolution": (720, 1280), "fps": 30.0, "caption": "ocean waves", "quality_score": 0.7},
    {"id": "h", "duration": 2.0, "resolution": (256, 256), "fps": 15.0, "caption": "minimal valid", "quality_score": 0.5},
]

valid, rejected = validate_clips(clips)
valid_ids = {c["id"] for c in valid}
rejected_ids = {c["id"] for c in rejected}
assert valid_ids == {"a", "d", "g", "h"}, f"Expected valid={{a,d,g,h}}, got {valid_ids}"
assert rejected_ids == {"b", "c", "e", "f"}, f"Expected rejected={{b,c,e,f}}, got {rejected_ids}"

# Batching
batches = batch_clips(valid, max_batch_duration=15.0)
# a(5) + d(10) = 15 fits; g(7) + h(2) = 9 fits
assert len(batches) == 2, f"Expected 2 batches, got {len(batches)}"
batch_durations = [sum(c["duration"] for c in b) for b in batches]
assert all(d <= 15.0 for d in batch_durations), f"Batch durations exceed max: {batch_durations}"

# Oversized single clip
big_clip = [{"id": "x", "duration": 20.0, "resolution": (720, 1280), "fps": 30.0, "caption": "long", "quality_score": 0.9}]
batches_big = batch_clips(big_clip, max_batch_duration=10.0)
assert len(batches_big) == 1, "Oversized clip should go in its own batch"
assert len(batches_big[0]) == 1

# Empty input
assert batch_clips([], max_batch_duration=10.0) == []

# Stats
stats = compute_batch_stats(batches)
assert len(stats) == 2
assert stats[0]["num_clips"] == 2
assert abs(stats[0]["total_duration"] - 15.0) < 1e-5
assert isinstance(stats[0]["resolutions"], set)
assert isinstance(stats[0]["avg_quality"], float)

print("Problem 3: ALL TESTS PASSED")

---

## Problem 4: Data Quality Report

### Scenario

You receive a batch of training records. Before adding them to the training set, you need to generate a quality report identifying issues.

### Requirements

**`audit_dataset(records: list[dict], schema: dict) -> dict`:**

The `schema` dict defines expected fields and their constraints:
```python
schema = {
    "id": {"type": str, "required": True},
    "caption": {"type": str, "required": True, "min_length": 5},
    "width": {"type": int, "required": True, "min": 256, "max": 4096},
    "height": {"type": int, "required": True, "min": 256, "max": 4096},
    "quality_score": {"type": float, "required": True, "min": 0.0, "max": 1.0},
    "source": {"type": str, "required": False},
}
```

Return a report dict with:
- `"total_records"`: int
- `"valid_records"`: int (records with zero issues)
- `"issues"`: list of `{"record_index": int, "field": str, "issue": str}` dicts
  - Issue types: `"missing"` (required field absent or None), `"wrong_type"` (wrong Python type), `"out_of_range"` (below min or above max), `"too_short"` (string shorter than min_length)
- `"duplicate_ids"`: list of ID values that appear more than once
- `"field_coverage"`: dict mapping each field name to the fraction of records that have it (0.0-1.0)

In [ ]:
def audit_dataset(records: list[dict], schema: dict) -> dict:
    """Audit a dataset against a schema and return a quality report."""
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 4 ---
schema = {
    "id": {"type": str, "required": True},
    "caption": {"type": str, "required": True, "min_length": 5},
    "width": {"type": int, "required": True, "min": 256, "max": 4096},
    "height": {"type": int, "required": True, "min": 256, "max": 4096},
    "quality_score": {"type": float, "required": True, "min": 0.0, "max": 1.0},
    "source": {"type": str, "required": False},
}

records = [
    {"id": "img1", "caption": "a red car on road", "width": 512, "height": 512, "quality_score": 0.9, "source": "web"},  # valid
    {"id": "img2", "caption": "ok", "width": 1024, "height": 768, "quality_score": 0.7},  # caption too short
    {"id": "img3", "caption": "a sunset over ocean", "width": 100, "height": 512, "quality_score": 0.8},  # width too small
    {"id": "img4", "caption": "a dog playing fetch", "width": 512, "height": 512, "quality_score": 1.5},  # quality out of range
    {"id": "img1", "caption": "duplicate id test", "width": 256, "height": 256, "quality_score": 0.6},  # duplicate id
    {"id": "img5", "caption": "valid image here", "width": 512, "height": 512, "quality_score": 0.5, "source": "synthetic"},  # valid
    {"id": "img6", "width": 512, "height": 512, "quality_score": 0.8},  # missing caption
    {"id": "img7", "caption": "a nice photo", "width": "not_int", "height": 512, "quality_score": 0.8},  # wrong type
]

report = audit_dataset(records, schema)

assert report["total_records"] == 8
assert report["valid_records"] == 2, f"Expected 2 valid, got {report['valid_records']}"  # img1 (first) and img5

# Check specific issues
issues_by_idx = {}
for issue in report["issues"]:
    issues_by_idx.setdefault(issue["record_index"], []).append(issue)

assert 1 in issues_by_idx  # img2: caption too short
assert any(i["issue"] == "too_short" for i in issues_by_idx[1])
assert 2 in issues_by_idx  # img3: width too small
assert any(i["issue"] == "out_of_range" and i["field"] == "width" for i in issues_by_idx[2])
assert 6 in issues_by_idx  # img6: missing caption
assert any(i["issue"] == "missing" for i in issues_by_idx[6])
assert 7 in issues_by_idx  # img7: wrong type for width
assert any(i["issue"] == "wrong_type" for i in issues_by_idx[7])

# Duplicate IDs
assert "img1" in report["duplicate_ids"]

# Field coverage
assert report["field_coverage"]["id"] == 1.0  # all have id
assert report["field_coverage"]["source"] == 2/8  # only 2 have source

# Empty input
empty_report = audit_dataset([], schema)
assert empty_report["total_records"] == 0
assert empty_report["valid_records"] == 0

print("Problem 4: ALL TESTS PASSED")

---

## Problem 5: Precision, Recall, F1 from Scratch

### Formulas (provided — you don't need to memorize these)

For each class `c`:
- **True Positives (TP):** items correctly predicted as class c
- **False Positives (FP):** items incorrectly predicted as class c (predicted c but actually another class)
- **False Negatives (FN):** items of class c that were predicted as another class

```
Precision(c) = TP / (TP + FP)    — of everything you predicted as c, how many were right?
Recall(c)    = TP / (TP + FN)     — of everything that IS c, how many did you find?
F1(c)        = 2 * P * R / (P + R) — harmonic mean (0 if P + R = 0)

Macro F1     = mean of per-class F1
Weighted F1  = sum(F1(c) * support(c)) / total_samples
              where support(c) = number of true instances of class c
```

### Requirements

**`compute_metrics(predictions: torch.Tensor, targets: torch.Tensor, num_classes: int) -> dict`:**
- Both are 1D int64 tensors
- Return dict with: `"per_class_precision"`, `"per_class_recall"`, `"per_class_f1"` (each a list of floats), `"macro_f1"` (float), `"weighted_f1"` (float)
- Handle edge cases: if a class has no predictions, precision = 0. If no ground truth, recall = 0.

In [ ]:
def compute_metrics(
    predictions: torch.Tensor, targets: torch.Tensor, num_classes: int
) -> dict:
    """Compute precision, recall, F1 per class, macro F1, and weighted F1."""
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 5 ---

# Binary case
preds = torch.tensor([0, 0, 1, 1, 1, 0])
targs = torch.tensor([0, 1, 1, 1, 0, 0])
result = compute_metrics(preds, targs, num_classes=2)

# Class 0: TP=2, FP=1, FN=1 → P=2/3, R=2/3, F1=2/3
# Class 1: TP=2, FP=1, FN=1 → P=2/3, R=2/3, F1=2/3
assert abs(result["per_class_precision"][0] - 2/3) < 1e-5
assert abs(result["per_class_recall"][0] - 2/3) < 1e-5
assert abs(result["per_class_f1"][0] - 2/3) < 1e-5
assert abs(result["macro_f1"] - 2/3) < 1e-5

# Multi-class
preds_mc = torch.tensor([0, 1, 2, 0, 1, 2, 0, 2])
targs_mc = torch.tensor([0, 1, 2, 0, 2, 1, 1, 2])
result_mc = compute_metrics(preds_mc, targs_mc, num_classes=3)
assert len(result_mc["per_class_f1"]) == 3
assert 0 <= result_mc["macro_f1"] <= 1
assert 0 <= result_mc["weighted_f1"] <= 1

# Edge: class with no predictions
preds_edge = torch.tensor([0, 0, 0])
targs_edge = torch.tensor([0, 1, 0])
result_edge = compute_metrics(preds_edge, targs_edge, num_classes=2)
assert result_edge["per_class_precision"][1] == 0.0  # no predictions for class 1

# Perfect predictions
perfect = compute_metrics(torch.tensor([0, 1, 2]), torch.tensor([0, 1, 2]), num_classes=3)
assert abs(perfect["macro_f1"] - 1.0) < 1e-5

print("Problem 5: ALL TESTS PASSED")

---

## Problem 6: PCA (Principal Component Analysis)

**Scenario:** Reduce the dimensionality of video frame embeddings for visualization and faster retrieval.

**Spec:**

Implement PCA from scratch:

1. **Center** the data: subtract the mean of each feature
2. **Covariance matrix:** $C = \frac{1}{N-1} X_c^T X_c$ where $X_c$ is centered data
3. **Eigendecomposition:** Use `torch.linalg.eigh(C)` — returns eigenvalues in ascending order
4. **Sort** eigenvectors by descending eigenvalue
5. **Project:** $X_{proj} = X_c @ V_k$ where $V_k$ is the top-k eigenvectors

**`PCA`:**
- `__init__(self, n_components: int)`
- `fit(self, X: torch.Tensor)` — X: (N, D). Compute and store mean, eigenvectors, eigenvalues.
- `transform(self, X: torch.Tensor) -> torch.Tensor` — project X into the learned PCA space. Return (N, n_components).
- `fit_transform(self, X: torch.Tensor) -> torch.Tensor` — fit and transform in one call.
- `explained_variance_ratio(self) -> torch.Tensor` — (n_components,) tensor. Each entry is eigenvalue_k / sum(all eigenvalues).
- `inverse_transform(self, X_proj: torch.Tensor) -> torch.Tensor` — reconstruct (approximately) from projected data. Return (N, D).

**Constraints:**
- No sklearn. Use `torch.linalg.eigh` for eigendecomposition.
- All operations in float32.

In [ ]:
class PCA:
    """Principal Component Analysis.

    Args:
        n_components: Number of principal components to keep.
    """

    def __init__(self, n_components: int):
        # YOUR CODE HERE
        pass

    def fit(self, X: torch.Tensor) -> None:
        # YOUR CODE HERE
        pass

    def transform(self, X: torch.Tensor) -> torch.Tensor:
        # YOUR CODE HERE
        pass

    def fit_transform(self, X: torch.Tensor) -> torch.Tensor:
        # YOUR CODE HERE
        pass

    def explained_variance_ratio(self) -> torch.Tensor:
        # YOUR CODE HERE
        pass

    def inverse_transform(self, X_proj: torch.Tensor) -> torch.Tensor:
        # YOUR CODE HERE
        pass

In [ ]:
# --- Tests: Problem 6 ---
torch.manual_seed(42)

# Data with clear principal direction
N, D = 200, 5
# First component has much more variance
X = torch.randn(N, D)
X[:, 0] *= 10.0  # first feature has 10x variance
X[:, 1] *= 5.0   # second has 5x

pca = PCA(n_components=2)
X_proj = pca.fit_transform(X)

# Shape check
assert X_proj.shape == (N, 2), f"Expected (200, 2), got {X_proj.shape}"

# Explained variance ratio should sum to < 1 and be sorted descending
evr = pca.explained_variance_ratio()
assert evr.shape == (2,), f"Expected (2,), got {evr.shape}"
assert evr[0] > evr[1], "First component should explain more variance"
assert evr.sum() < 1.0 + 1e-5, "Ratios should sum to <= 1"
assert evr[0] > 0.5, f"First component should explain > 50% variance, got {evr[0]:.3f}"

# Transform gives same result as fit_transform
pca2 = PCA(n_components=2)
pca2.fit(X)
X_proj2 = pca2.transform(X)
assert torch.allclose(X_proj, X_proj2, atol=1e-4)

# Inverse transform: reconstruction should be close for high-variance components
X_reconstructed = pca.inverse_transform(X_proj)
assert X_reconstructed.shape == (N, D), f"Expected ({N}, {D}), got {X_reconstructed.shape}"
# Reconstruction error should be small for top 2 components (which capture most variance)
recon_error = ((X - X_reconstructed) ** 2).mean().item()
total_var = X.var().item() * D
assert recon_error < total_var * 0.3, f"Reconstruction error too high: {recon_error:.2f}"

# n_components = D should give perfect reconstruction
pca_full = PCA(n_components=D)
X_full = pca_full.fit_transform(X)
X_recon_full = pca_full.inverse_transform(X_full)
assert torch.allclose(X, X_recon_full, atol=1e-3), "Full PCA should reconstruct perfectly"

# Projected data should be uncorrelated (covariance matrix is ~diagonal)
cov_proj = torch.cov(X_proj.T)
off_diag = cov_proj[0, 1].abs().item()
assert off_diag < 0.5, f"Projected features should be uncorrelated, off-diag cov = {off_diag:.3f}"

# n_components = 1
pca_1 = PCA(n_components=1)
X_1 = pca_1.fit_transform(X)
assert X_1.shape == (N, 1)

print("Problem 6: ALL TESTS PASSED")

---

## Problem 7: Naive Bayes Classifier (Gaussian)

**Scenario:** Fast baseline for classifying data points using probabilistic modeling. Good for text/feature classification when features are roughly independent.

**Spec:**

Implement Gaussian Naive Bayes:

**Training:** For each class $c$:
- Compute prior: $P(c) = \frac{N_c}{N}$
- Compute per-feature mean $\mu_{c,d}$ and variance $\sigma^2_{c,d}$

**Prediction:** For each query point $x$:
$$\log P(c|x) \propto \log P(c) + \sum_d \log P(x_d | c)$$
$$\log P(x_d | c) = -\frac{1}{2} \log(2\pi\sigma^2_{c,d}) - \frac{(x_d - \mu_{c,d})^2}{2\sigma^2_{c,d}}$$

Add a small `eps=1e-6` to variances to avoid division by zero. Predict the class with highest log-posterior.

**`GaussianNaiveBayes`:**
- `__init__(self)`
- `fit(self, X: torch.Tensor, y: torch.Tensor)` — X: (N, D), y: (N,) int64
- `predict(self, X: torch.Tensor) -> torch.Tensor` — return (M,) int64 predicted labels
- `predict_log_proba(self, X: torch.Tensor) -> torch.Tensor` — return (M, C) log-probabilities (unnormalized is fine)

**Constraints:**
- No sklearn. Pure tensor operations.
- Use vectorized computation (no loops over features).

In [ ]:
class GaussianNaiveBayes:
    """Gaussian Naive Bayes classifier."""

    def __init__(self):
        # YOUR CODE HERE
        pass

    def fit(self, X: torch.Tensor, y: torch.Tensor) -> None:
        # YOUR CODE HERE
        pass

    def predict(self, X: torch.Tensor) -> torch.Tensor:
        # YOUR CODE HERE
        pass

    def predict_log_proba(self, X: torch.Tensor) -> torch.Tensor:
        # YOUR CODE HERE
        pass

In [ ]:
# --- Tests: Problem 7 ---
torch.manual_seed(42)

# Well-separated Gaussian clusters
X_train = torch.cat([
    torch.randn(100, 3) + torch.tensor([5.0, 0.0, 0.0]),   # class 0
    torch.randn(100, 3) + torch.tensor([0.0, 5.0, 0.0]),   # class 1
    torch.randn(100, 3) + torch.tensor([0.0, 0.0, 5.0]),   # class 2
])
y_train = torch.cat([torch.zeros(100), torch.ones(100), torch.full((100,), 2)]).long()

gnb = GaussianNaiveBayes()
gnb.fit(X_train, y_train)

# Predict on training data: should be highly accurate
preds = gnb.predict(X_train)
assert preds.shape == (300,)
assert preds.dtype == torch.int64
accuracy = (preds == y_train).float().mean().item()
assert accuracy > 0.95, f"Accuracy on well-separated data should be > 95%, got {accuracy:.1%}"

# Log probabilities shape
log_proba = gnb.predict_log_proba(X_train)
assert log_proba.shape == (300, 3), f"Expected (300, 3), got {log_proba.shape}"

# Predictions should match argmax of log_proba
assert (preds == log_proba.argmax(dim=1)).all(), "Predictions should match argmax of log_proba"

# Test on new points clearly in each cluster
test_points = torch.tensor([
    [5.0, 0.0, 0.0],   # should be class 0
    [0.0, 5.0, 0.0],   # should be class 1
    [0.0, 0.0, 5.0],   # should be class 2
])
test_preds = gnb.predict(test_points)
assert test_preds.tolist() == [0, 1, 2], f"Expected [0,1,2], got {test_preds.tolist()}"

# Binary case
X_bin = torch.cat([torch.randn(50, 2) - 2, torch.randn(50, 2) + 2])
y_bin = torch.cat([torch.zeros(50), torch.ones(50)]).long()
gnb_bin = GaussianNaiveBayes()
gnb_bin.fit(X_bin, y_bin)
acc_bin = (gnb_bin.predict(X_bin) == y_bin).float().mean().item()
assert acc_bin > 0.90, f"Binary accuracy should be > 90%, got {acc_bin:.1%}"

print("Problem 7: ALL TESTS PASSED")

---

## Problem 8: Simple Training Loop + Early Stopping

### Requirements

**`Trainer`:**
- `__init__(self, model, optimizer, train_loader, val_loader)`
- `train_epoch(self) -> float` — run one training epoch, return mean loss
  - For each batch: `forward → loss → backward → optimizer.step() → optimizer.zero_grad()`
  - The model's `forward` takes a batch (tuple of tensors) and returns `(loss, logits)`
  - Set model to train mode
- `evaluate(self) -> float` — run validation, return mean loss
  - No gradient computation (`torch.no_grad()`)
  - Set model to eval mode
- `fit(self, num_epochs: int, patience: int) -> dict` — train with early stopping
  - After each epoch, evaluate on validation set
  - If val loss hasn't improved for `patience` consecutive epochs, stop early
  - Track the best val loss and which epoch it occurred at (0-indexed)
  - Return:
    ```python
    {
        "train_losses": list[float],
        "val_losses": list[float],
        "best_epoch": int,
        "stopped_early": bool
    }
    ```

In [ ]:
class Trainer:
    """Training loop with early stopping."""

    def __init__(self, model: nn.Module, optimizer: torch.optim.Optimizer,
                 train_loader: DataLoader, val_loader: DataLoader):
        # YOUR CODE HERE
        pass

    def train_epoch(self) -> float:
        # YOUR CODE HERE
        pass

    def evaluate(self) -> float:
        # YOUR CODE HERE
        pass

    def fit(self, num_epochs: int, patience: int) -> dict:
        # YOUR CODE HERE
        pass

In [ ]:
# --- Tests for Problem 8 ---

# Simple regression model for testing
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(10, 1)
    def forward(self, batch):
        x, y = batch
        pred = self.linear(x).squeeze(-1)
        loss = F.mse_loss(pred, y)
        return loss, pred

torch.manual_seed(42)
model = SimpleModel()

# Create data with learnable pattern
X = torch.randn(200, 10)
true_w = torch.randn(10)
y = X @ true_w + 0.1 * torch.randn(200)

train_ds = torch.utils.data.TensorDataset(X[:160], y[:160])
val_ds = torch.utils.data.TensorDataset(X[160:], y[160:])
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
trainer = Trainer(model, optimizer, train_loader, val_loader)

# Single epoch
train_loss = trainer.train_epoch()
assert isinstance(train_loss, float), "train_epoch should return float"
assert train_loss > 0, "Loss should be positive"

val_loss = trainer.evaluate()
assert isinstance(val_loss, float), "evaluate should return float"
assert val_loss > 0, "Val loss should be positive"

# Full training with early stopping
torch.manual_seed(42)
model2 = SimpleModel()
optimizer2 = torch.optim.Adam(model2.parameters(), lr=0.01)
trainer2 = Trainer(model2, optimizer2, train_loader, val_loader)

result = trainer2.fit(num_epochs=50, patience=5)
assert "train_losses" in result
assert "val_losses" in result
assert "best_epoch" in result
assert "stopped_early" in result
assert len(result["train_losses"]) == len(result["val_losses"])

# Loss should decrease
assert result["train_losses"][-1] < result["train_losses"][0], "Training loss should decrease"

# If stopped early, should have run fewer than 50 epochs
if result["stopped_early"]:
    assert len(result["train_losses"]) < 50
    assert len(result["train_losses"]) <= result["best_epoch"] + 5 + 1

# best_epoch should be valid
assert 0 <= result["best_epoch"] < len(result["val_losses"])
best_val = result["val_losses"][result["best_epoch"]]
assert best_val == min(result["val_losses"]), "best_epoch should have the lowest val loss"

print("Problem 8: ALL TESTS PASSED")

---

## Scoring Yourself

### Warmups (Part A)

| Result | Assessment |
|--------|------------|
| < 5 min, first try | **Automatic** — this pattern is locked in |
| 5-10 min | **Solid** — you'd be fine |
| 10-15 min or needed to debug | **Shaky** — drill this one again tomorrow |
| Couldn't solve | **Gap** — review the concept, then redo |

### Full Problems (Part B)

| Result | Assessment |
|--------|------------|
| Solved in < 25 min, all tests pass | **Strong pass** |
| Solved in 25-30 min, minor debugging | **Pass** — exam-ready |
| 30-40 min or needed hints | **Borderline** — redo tomorrow |
| Couldn't solve or > 40 min | **Drill this** — break it into pieces |